# Week 8 (Tue) - Class Activities in CS 315 (DuckDB Version)

Solving tasks from `netflix_data.db` using DuckDB SQL queries.

**Table of Contents**

0. [Q0: Connect to Database & List Tables](#q0)
1. [Q1: Inspect Table Data](#q1)
2. [Q2: Show Number of Rows](#q2)
3. [Q3: Show Column Names](#q3)
4. [Q4: Show Number of Total Countries](#q4)
5. [Q5: Show Number of TV Series](#q5)
6. [Q6: Number of TV Series in South Korea](#q6)
7. [Q7: Total Weeks of Squid Game in Italy](#q7)
8. [Q8: Time Frame for a Given Country](#q8)
9. [Q9: Is Number of Weeks in Ranking Equal?](#q9)
10. [Q10: Country with Fewest Weeks](#q10)

<a id="q0"></a>
### Q0: Connect to Database & List Tables

Connect to `netflix_data.db` and inspect the available tables in the database.

In [2]:
import duckdb

con = duckdb.connect('netflix_data.db')
con

When we open a connection it will remain open until we explicetely close it.

We can execute simple SQL commands as shown below.

In [3]:
con.sql("SHOW TABLES")

┌──────────┐
│   name   │
│ varchar  │
├──────────┤
│ films    │
│ tvseries │
└──────────┘

<a id="q1"></a>
### Q1: Load & Preview Data

Preview the top rows `tvseries`. This is similar to the `.head` method in pandas.

In [4]:
con.sql("SELECT * FROM tvseries LIMIT 5")

┌──────────────┬──────────────┬────────────┬─────────────┬──────────────────────┬───────────────────────────────────┬────────────────────────────┐
│ country_name │ country_iso2 │    week    │ weekly_rank │      show_title      │           season_title            │ cumulative_weeks_in_top_10 │
│   varchar    │   varchar    │  varchar   │    int64    │       varchar        │              varchar              │           int64            │
├──────────────┼──────────────┼────────────┼─────────────┼──────────────────────┼───────────────────────────────────┼────────────────────────────┤
│ Argentina    │ AR           │ 2026-10-04 │           1 │ Not a Stranger       │ Not a Stranger: Season 1          │                          3 │
│ Argentina    │ AR           │ 2026-10-04 │           2 │ My Sad Dead          │ My Sad Dead: Limited Series       │                          2 │
│ Argentina    │ AR           │ 2026-10-04 │           3 │ The Final Problem    │ The Final Problem: Limited Series │ 

<a id="q2"></a>
### Q2: Show number of rows

In [5]:
con.sql("SELECT COUNT(*) FROM tvseries")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       256100 │
└──────────────┘

Because the result is a table, if we want just the number, we will use the method `fetchone` to get the first row:

In [6]:
con.sql("SELECT COUNT(*) FROM tvseries").fetchone()

(256100,)

Since the result is a tuple, then we can index it like this:

In [7]:
con.sql("SELECT COUNT(*) FROM tvseries").fetchone()[0]

256100

<a id="q3"></a>
### Q3: Show column names

In [8]:
con.sql("DESCRIBE tvseries")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ country_name               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country_iso2               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ week                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ weekly_rank                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ show_title                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ season_title               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cumulative_weeks_in_top_10 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────────┴─────────────┴─────────┴─────────┴─

<a id="q4"></a>
### Q4: Show number of total countries

In [9]:
con.sql("SELECT COUNT(DISTINCT country_name) FROM tvseries").fetchone()[0]

94

<a id="q5"></a>
### Q5: Show number of TV series

In [10]:
con.sql("SELECT COUNT(DISTINCT show_title) FROM tvseries").fetchone()[0]

3574

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [11]:
con.sql("""
    SELECT COUNT(DISTINCT show_title) 
    FROM tvseries 
    WHERE country_name = 'South Korea'
""").fetchone()[0]

481

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [12]:
con.sql("""
    SELECT COUNT(DISTINCT week) 
    FROM tvseries 
    WHERE country_name = 'Italy' AND show_title = 'Squid Game'
""").fetchone()[0]

29

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [13]:
def show_timeframe(con, country):
    """A function to print out the time frame a country was in the Netflix rankings using DuckDB."""
    res = con.sql("""
        SELECT MIN(week), MAX(week)
        FROM (
            SELECT country_name, week FROM tvseries
        )
        WHERE country_name = $1
    """, params=[country]).fetchone()
    
    start, end = res[0], res[1]
    print(f"Netflix rankings for {country} exist between {start} and {end}.")

show_timeframe(con, "Chile")
show_timeframe(con, "Russia")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-04.
Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27.


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

Check if total rows match `20 * unique_weeks * unique_countries`.

In [14]:
is_equal = con.sql("""
    SELECT COUNT(*) = 10 * COUNT(DISTINCT week) * COUNT(DISTINCT country_name) 
    FROM tvseries
""").fetchone()[0]

is_equal

False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Find the country with the fewest weeks in the Netflix rankings.

In [15]:
con.sql("""
SELECT country_name, COUNT(DISTINCT week) AS weekCnt
FROM tvseries
GROUP BY country_name
ORDER BY weekCnt ASC
LIMIT 5;""")

┌──────────────┬─────────┐
│ country_name │ weekCnt │
│   varchar    │  int64  │
├──────────────┼─────────┤
│ Russia       │      35 │
│ Denmark      │     275 │
│ Kuwait       │     275 │
│ Hungary      │     275 │
│ Nicaragua    │     275 │
└──────────────┴─────────┘

## Week 8 Task 3C

In [19]:
import pandas as pd
df = pd.read_csv("/Users/wuyuwei/Desktop/all-weeks-countries.tsv", sep="\t")
target_kdramas = ["True Beauty", "My ID is Gangnam Beauty"]
kdrama_df = df[df["show_title"].isin(target_kdramas)]

con = duckdb.connect("simple_netflix.db")
con.execute("CREATE OR REPLACE TABLE kdramas AS SELECT * FROM kdrama_df")

print(con.sql("""
    SELECT country_name, week, show_title, weekly_rank, cumulative_weeks_in_top_10 
    FROM kdramas 
"""))

con.close()

┌──────────────┬────────────┬─────────────┬─────────────┬────────────────────────────┐
│ country_name │    week    │ show_title  │ weekly_rank │ cumulative_weeks_in_top_10 │
│   varchar    │  varchar   │   varchar   │    int64    │           int64            │
├──────────────┼────────────┼─────────────┼─────────────┼────────────────────────────┤
│ Argentina    │ 2024-01-14 │ True Beauty │           9 │                          5 │
│ Argentina    │ 2024-01-07 │ True Beauty │           5 │                          4 │
│ Argentina    │ 2023-12-31 │ True Beauty │           5 │                          3 │
│ Argentina    │ 2023-12-24 │ True Beauty │           4 │                          2 │
│ Argentina    │ 2023-12-17 │ True Beauty │           3 │                          1 │
│ Australia    │ 2023-02-19 │ True Beauty │           3 │                          2 │
│ Australia    │ 2023-02-12 │ True Beauty │           5 │                          1 │
│ Bahrain      │ 2023-04-16 │ True Beauty │